# XGBoost Regressor Notebook (Using CRSP Data)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
import optuna
from sklearn.metrics import mean_absolute_error

c:\Users\jgte2\anaconda3\envs\cljc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NUM_COL = 'dlyclose'
DATE_COL = 'dlycaldt'
TICKER_COL = 'permno'
SPLIT_DATE = "2014-01-01"
START_BALANCE = 1000
EXIT_PROP = 1.0

##### Data Prep

In [3]:
data = pd.read_csv('./data/crsp_dsf.csv')
data[DATE_COL] = data[DATE_COL].astype('datetime64[s]')
data = data.dropna(subset=NUM_COL).reset_index(drop=True)
data.head()

C:\Users\jgte2\AppData\Local\Temp\ipykernel_2832\2761941887.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('./data/crsp_dsf.csv')


MemoryError: Unable to allocate 2.52 GiB for an array with shape (22, 15361703) and data type object

In [ ]:
reg_df = data.sort_values([TICKER_COL, DATE_COL]).copy()

reg_df["target"] = (reg_df.groupby(TICKER_COL)[NUM_COL].shift(-1))

for lag in [1, 2, 3, 5, 10, 20]:
    reg_df[f"lag_prc_{lag}"] = (reg_df.groupby(TICKER_COL)[NUM_COL].shift(lag))

for lag in [1, 2, 3, 5, 10]:
    reg_df[f"lag_ret_{lag}"] = (reg_df.groupby(TICKER_COL)["ret"].shift(lag))

for win in [5, 10, 20, 50]:
    reg_df[f"ma_{win}"] = (reg_df.groupby(TICKER_COL)[NUM_COL].transform(lambda x: x.rolling(win).mean()))

reg_df["vol_20"] = (reg_df.groupby(TICKER_COL)["ret"].transform(lambda x: x.rolling(20).std()))

reg_df["vol_ma_20"] = (reg_df.groupby(TICKER_COL)["vol"].transform(lambda x: x.rolling(20).mean()))

reg_df["vol_ratio"] = reg_df["vol"] / reg_df["vol_ma_20"]

reg_df["spread"] = reg_df["ask"] - reg_df["bid"]

FEATURES = ["lag_prc_1", "lag_prc_2", "lag_prc_3", "lag_prc_5", "lag_prc_10", "lag_prc_20",
            "lag_ret_1", "lag_ret_2", "lag_ret_3", "lag_ret_5", "lag_ret_10",
            "ma_5", "ma_10", "ma_20", "ma_50", "vol_20", "vol_ratio", "spread", "vol"]

reg_df = reg_df[[TICKER_COL, DATE_COL, NUM_COL, "target"] + FEATURES].dropna().reset_index(drop=True)

train_df = reg_df[reg_df[DATE_COL] < SPLIT_DATE]
test_df = reg_df[reg_df[DATE_COL] >= SPLIT_DATE]

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

reg_df.head()

##### Basline Model

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0,
    n_jobs=-1
)

model.fit(X_train, y_train)
preds = model.predict(X_test)

In [ ]:
preds_df = test_df[[DATE_COL, TICKER_COL, NUM_COL, "target"]].copy()
preds_df['pred'] = preds

##### Trading Strategy

In [ ]:
def trade_strat_single(df, curr_b=1000, exit_prop=1.0, pred_col='pred'):
    ### Defaults to starting balance of 1000 and full exits
    curr_vol = 0

    returns = []

    for _, row in df.iterrows():

        pred = row[pred_col]
        num = row[NUM_COL]

        ### Buy signal
        if pred >= num:
            ### Only buy if we have balance
            ### Otherwise we will just be holding the stock
            if curr_b > 0:
                curr_vol += curr_b / num
                curr_b = 0

        ### Sell signal
        else:
            ### Only sell if we have volume to sell
            if curr_vol > 0:
                shares_sold = exit_prop * curr_vol
                curr_b += shares_sold * num
                curr_vol -= shares_sold
                
        ### Current Return of the Strategy
        ### Cash Balance + Value of Held Stock
        returns.append(curr_b + curr_vol * num)
    
    return returns

In [ ]:
def trade_strat(data, tickers, curr_b=START_BALANCE, exit_prop=EXIT_PROP, pred_col='pred'):
    ret_df = pd.DataFrame()
    for t in tickers:
        df = data[data[TICKER_COL] == t].reset_index(drop=True)
        returns = trade_strat_single(df, curr_b=curr_b, exit_prop=exit_prop, pred_col=pred_col)
        start_vol = START_BALANCE / df[NUM_COL].iloc[0]
        hold_ret = start_vol * df[NUM_COL].values
        stg_df = (
            df[[TICKER_COL, DATE_COL]]
            .copy()
            .assign(
                strat_ret=returns,
                hold_ret=hold_ret
            )
        )
        if len(ret_df) == 0:
            ret_df = stg_df.copy()
        else:
            ret_df = pd.concat([ret_df, stg_df])
    
    return ret_df

In [ ]:
# tickers = preds_df[TICKER_COL].unique()
# ret_df = trade_strat(preds_df, tickers)
# ret_df.to_csv(f"./data/xgboost_reg_baseline_model_returns.csv", index=False)

ret_df = pd.read_csv(f"./data/xgboost_reg_baseline_model_returns.csv")
ret_df[DATE_COL] = pd.to_datetime(ret_df[DATE_COL])

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum"}).iloc[:-1,:]

sns.lineplot(x=plot_df.index, y=np.log(plot_df['hold_ret']), label="Long Hold")
sns.lineplot(x=plot_df.index, y=np.log(plot_df['strat_ret']), label="Baseline Model Strategy")

plt.title("Portfolio Returns Comparison - Baseline Model Strategy v. Long Hold")
plt.ylabel("Log($ Return)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum", TICKER_COL : "count"}).rename(columns={TICKER_COL:"num_stocks"}).iloc[:-1,:]
plot_df['base_bal'] = plot_df["num_stocks"] * START_BALANCE
plot_df["strat_pct_grwth"] = (plot_df["strat_ret"] / plot_df['base_bal'] - 1) * 100
plot_df["hold_pct_grwth"] = (plot_df["hold_ret"] / plot_df['base_bal'] - 1) * 100

sns.lineplot(x=plot_df.index, y=plot_df['hold_pct_grwth'], label="Long Hold")
sns.lineplot(x=plot_df.index, y=plot_df['strat_pct_grwth'], label="Baseline Model Strategy")

plt.title("Portfolio Returns Comparison - Baseline Model Strategy v. Long Hold")
plt.ylabel("% Return")
plt.xticks(rotation=45)
plt.legend()
plt.show()

##### Model Fine-Tuning

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }

    model = XGBRegressor(
        **params,
        random_state=0,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return mean_absolute_error(y_test, preds)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

best_params = study.best_params

In [ ]:
final_model = XGBRegressor(
    **best_params,
    random_state=0,
    n_jobs=-1
)

final_model.fit(X_train, y_train)
final_preds = final_model.predict(X_test)

In [ ]:
try:
    preds_df['final_pred'] = final_preds
except:
    preds_df = test_df[[DATE_COL, TICKER_COL, NUM_COL, "target"]].copy()
    preds_df['final_pred'] = final_preds

In [ ]:
tickers = preds_df[TICKER_COL].unique()
ret_df = trade_strat(preds_df, tickers, pred_col='final_pred')
ret_df.to_csv(f"./data/xgboost_reg_tuned_model_returns.csv", index=False)

# ret_df = pd.read_csv(f"./data/xgboost_reg_tuned_model_returns.csv")
# ret_df[DATE_COL] = pd.to_datetime(ret_df[DATE_COL])

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum"}).iloc[:-1,:]

sns.lineplot(x=plot_df.index, y=np.log(plot_df['hold_ret']), label="Long Hold")
sns.lineplot(x=plot_df.index, y=np.log(plot_df['strat_ret']), label="Tuned Model Strategy")

plt.title("Portfolio Returns Comparison - Tuned Model Strategy v. Long Hold")
plt.ylabel("Log($ Return)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
### There was some weird data corruption on the last day
plot_df = ret_df.groupby(by=DATE_COL).agg({"strat_ret" : "sum", "hold_ret" : "sum", TICKER_COL : "count"}).rename(columns={TICKER_COL:"num_stocks"}).iloc[:-1,:]
plot_df['base_bal'] = plot_df["num_stocks"] * START_BALANCE
plot_df["strat_pct_grwth"] = (plot_df["strat_ret"] / plot_df['base_bal'] - 1) * 100
plot_df["hold_pct_grwth"] = (plot_df["hold_ret"] / plot_df['base_bal'] - 1) * 100

sns.lineplot(x=plot_df.index, y=plot_df['hold_pct_grwth'], label="Long Hold")
sns.lineplot(x=plot_df.index, y=plot_df['strat_pct_grwth'], label="Tuned Model Strategy")

plt.title("Portfolio Returns Comparison - Tuned Model Strategy v. Long Hold")
plt.ylabel("% Return")
plt.xticks(rotation=45)
plt.legend()
plt.show()